In [1]:
# ============================================================
# TASK 23 : HARDENING, SCALE & MLOPS
# REGISTRY + FEATURE STORE
# ============================================================

"""
OBJECTIVE

Build a production-ready Feature Store and Feature Registry
for the job recommendation system.

Definition of Done

✓ Feature Store Created
✓ Feature Registry Available
✓ Real Dataset Used
✓ Explainable Features
✓ Ready for Model Training
"""

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import datetime

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns",None)
pd.set_option("display.width",200)

# ============================================================
# LOAD DATASETS
# ============================================================

students = pd.read_csv("../datasets/students.csv")
jobs = pd.read_csv("../datasets/jobs.csv")
matches = pd.read_csv("../datasets/matches.csv")

print("="*90)
print("REAL DATASETS LOADED")
print("="*90)

print(f"Students Dataset : {students.shape}")
print(f"Jobs Dataset     : {jobs.shape}")
print(f"Matches Dataset  : {matches.shape}")

# ============================================================
# MERGE DATASETS
# ============================================================

data = matches.merge(
    students,
    on="student_id"
)

data = data.merge(
    jobs,
    on="job_id"
)

print("\nMerged Dataset Shape :", data.shape)

# ============================================================
# FEATURE ENGINEERING
# ============================================================

data["location_match"] = (
    data["location_x"] == data["location_y"]
).astype(int)

data["role_match"] = (
    data["preferred_role"] == data["job_title"]
).astype(int)

data["experience_score"] = (
    1 - (
        data["experience_gap"] /
        data["experience_gap"].max()
    )
)

data["skill_density"] = (
    data["skill_overlap_count"] /
    (data["skill_overlap_count"].max() + 1)
)

data["combined_score"] = (
    0.60 * data["skill_overlap_ratio"] +
    0.40 * data["experience_score"]
)

# Extra engineered features

data["experience_level"] = np.where(
    data["internship_months"] >= 18,
    1,
    0
)

data["certification_count"] = (
    data["certifications"]
    .fillna("")
    .astype(str)
    .apply(lambda x: len(x.split(",")))
)

data["education_score"] = (
    data["education_level"]
    .map({
        "Diploma":1,
        "BE":2,
        "BTech":3,
        "MCA":4,
        "MTech":5
    })
    .fillna(0)
)

# ============================================================
# FEATURE STORE
# ============================================================

feature_store = data[[
    "student_id",
    "job_id",
    "skill_overlap_count",
    "skill_overlap_ratio",
    "experience_gap",
    "experience_score",
    "location_match",
    "role_match",
    "skill_density",
    "combined_score",
    "experience_level",
    "certification_count",
    "education_score",
    "label"
]].copy()

feature_store["feature_store_version"] = "v1.0"
feature_store["created_at"] = datetime.datetime.now()

print("\n")
print("="*90)
print("FEATURE STORE CREATED")
print("="*90)

print(f"Total Feature Records : {len(feature_store)}")

display(feature_store.head())

# ============================================================
# FEATURE REGISTRY
# ============================================================

feature_registry = pd.DataFrame({

    "Feature":[
        "skill_overlap_count",
        "skill_overlap_ratio",
        "experience_gap",
        "experience_score",
        "location_match",
        "role_match",
        "skill_density",
        "combined_score",
        "experience_level",
        "certification_count",
        "education_score"
    ],

    "Type":[
        "Integer",
        "Float",
        "Float",
        "Float",
        "Binary",
        "Binary",
        "Float",
        "Float",
        "Binary",
        "Integer",
        "Ordinal"
    ],

    "Source":[
        "matches.csv",
        "matches.csv",
        "matches.csv",
        "Engineered",
        "students + jobs",
        "students + jobs",
        "Engineered",
        "Engineered",
        "students.csv",
        "students.csv",
        "students.csv"
    ],

    "Description":[
        "Number of matching skills",
        "Skill overlap percentage",
        "Experience difference",
        "Normalized experience score",
        "Same location indicator",
        "Preferred role matches job",
        "Normalized skill density",
        "Weighted recommendation score",
        "Internship experience category",
        "Total certifications",
        "Education level score"
    ]

})

print("\n")
print("="*90)
print("FEATURE REGISTRY")
print("="*90)

display(feature_registry)

print("\n")

print("✓ Feature Store Ready")
print("✓ Feature Registry Ready")
print("✓ MLOps Foundation Initialized")

REAL DATASETS LOADED
Students Dataset : (20, 7)
Jobs Dataset     : (9, 7)
Matches Dataset  : (180, 6)

Merged Dataset Shape : (180, 18)


FEATURE STORE CREATED
Total Feature Records : 180


,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,experience_score,location_match,role_match,skill_density,combined_score,experience_level,certification_count,education_score,label,feature_store_version,created_at
0,1,101,3,1.000,2.0,0.6,1,1,0.75,0.8400,1,2,3,1,v1.0,2026-07-10 22:21:21.900302
1,1,102,1,0.333,1.0,0.8,0,0,0.25,0.5198,1,2,3,0,v1.0,2026-07-10 22:21:21.900302
2,1,103,1,0.333,2.0,0.6,0,0,0.25,0.4398,1,2,3,0,v1.0,2026-07-10 22:21:21.900302
3,1,104,2,0.667,2.0,0.6,1,0,0.50,0.6402,1,2,3,1,v1.0,2026-07-10 22:21:21.900302
4,1,105,0,0.000,2.0,0.6,0,0,0.00,0.2400,1,2,3,0,v1.0,2026-07-10 22:21:21.900302




FEATURE REGISTRY


,Feature,Type,Source,Description
0,skill_overlap_count,Integer,matches.csv,Number of matching skills
1,skill_overlap_ratio,Float,matches.csv,Skill overlap percentage
2,experience_gap,Float,matches.csv,Experience difference
3,experience_score,Float,Engineered,Normalized experience score
4,location_match,Binary,students + jobs,Same location indicator
5,role_match,Binary,students + jobs,Preferred role matches job
6,skill_density,Float,Engineered,Normalized skill density
7,combined_score,Float,Engineered,Weighted recommendation score
8,experience_level,Binary,students.csv,Internship experience category
9,certification_count,Integer,students.csv,Total certifications




✓ Feature Store Ready
✓ Feature Registry Ready
✓ MLOps Foundation Initialized


In [ ]:
# ============================================================
# MODEL TRAINING + MODEL REGISTRY
# ============================================================

print("="*90)
print("MODEL TRAINING")
print("="*90)

# ------------------------------------------------------------
# Feature Matrix
# ------------------------------------------------------------

FEATURE_COLUMNS=[

    "skill_overlap_count",
    "skill_overlap_ratio",
    "experience_gap",
    "experience_score",
    "location_match",
    "role_match",
    "skill_density",
    "combined_score",
    "experience_level",
    "certification_count",
    "education_score"

]

X=feature_store[FEATURE_COLUMNS]

y=feature_store["label"]

# ------------------------------------------------------------
# Train Test Split
# ------------------------------------------------------------

X_train,X_test,y_train,y_test=train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

print(f"Training Records : {len(X_train)}")
print(f"Testing Records  : {len(X_test)}")

# ============================================================
# BASELINE MODEL
# ============================================================

print("\nCreating Baseline Model...")

baseline_model=RandomForestClassifier(

    random_state=42

)

baseline_model.fit(

    X_train,

    y_train

)

baseline_accuracy=baseline_model.score(

    X_test,

    y_test

)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")

# ============================================================
# HYPERPARAMETER SEARCH
# ============================================================

print("\nRunning Grid Search...")

parameter_grid={

    "n_estimators":[300,500],

    "max_depth":[10,15,None],

    "min_samples_split":[2,3],

    "min_samples_leaf":[1,2],

    "max_features":["sqrt"],

    "class_weight":["balanced"]

}

grid=GridSearchCV(

    estimator=RandomForestClassifier(

        random_state=42

    ),

    param_grid=parameter_grid,

    cv=5,

    scoring="f1",

    n_jobs=-1,

    verbose=1

)

grid.fit(

    X_train,

    y_train

)

model=grid.best_estimator_

print("\nBest Parameters")

for key,value in grid.best_params_.items():

    print(f"{key:20}: {value}")

print(f"\nBest CV Score : {grid.best_score_:.4f}")

# ============================================================
# CROSS VALIDATION
# ============================================================

cv=StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

cv_scores=cross_val_score(

    model,

    X,

    y,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)

print("\nCross Validation Scores")

print(cv_scores)

print(f"\nAverage Accuracy : {cv_scores.mean():.4f}")

# ============================================================
# FINAL TRAINING
# ============================================================

model.fit(

    X_train,

    y_train

)

print("\n✓ Final Model Successfully Trained")

# ============================================================
# MODEL REGISTRY
# ============================================================

model_registry=pd.DataFrame({

    "Model Name":[
        "RandomForest Recommendation Model"
    ],

    "Version":[
        "v1.0"
    ],

    "Algorithm":[
        "RandomForestClassifier"
    ],

    "Training Records":[
        len(X_train)
    ],

    "Testing Records":[
        len(X_test)
    ],

    "CV Accuracy":[
        round(cv_scores.mean(),4)
    ],

    "Status":[
        "Production Ready"
    ]

})

print("\n")
print("="*90)
print("MODEL REGISTRY")
print("="*90)

display(model_registry)

# ============================================================
# FEATURE STORE VALIDATION
# ============================================================

print("\n")
print("="*90)
print("FEATURE STORE VALIDATION")
print("="*90)

print(f"Total Features        : {len(FEATURE_COLUMNS)}")
print(f"Feature Store Records : {len(feature_store)}")
print(f"Registry Entries      : {len(feature_registry)}")

missing=feature_store.isnull().sum().sum()

duplicates=feature_store.duplicated().sum()

print(f"Missing Values        : {missing}")
print(f"Duplicate Records     : {duplicates}")

if missing==0 and duplicates==0:

    print("\n✓ Feature Store Validation Passed")

else:

    print("\n⚠ Feature Store Requires Review")

print("\n")
print("✓ Feature Store Ready")
print("✓ Model Registry Ready")
print("✓ Model Ready For Evaluation")

MODEL TRAINING
Training Records : 144
Testing Records  : 36

Creating Baseline Model...
Baseline Accuracy : 1.0000

Running Grid Search...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
